# 材料一：协同过滤推荐系统

在这份练习中，你将通过实现协同过滤，建立一个电影推荐系统



![Image Name](https://cdn.kesci.com/upload/image/rgftw1vlig.png?imageView2/0/w/320/h/320)





##  包


我们将使用现在比较熟悉的Numpy和Tensorflow包

In [7]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from recsys_utils import *
tf.enable_eager_execution()

AttributeError: module 'tensorflow' has no attribute 'enable_eager_execution'

## 1 - 符号说明


|General Notation  | Description | Python (if any) |
|:-------------|:------------------------------------------------------------|:------|
| $r(i,j)$     | scalar; = 1  if user j rated game i  = 0  otherwise             ||
| $y(i,j)$     | scalar; = rating given by user j on game  i    (if r(i,j) = 1 is defined) ||
|$\mathbf{w}^{(j)}$ | vector; parameters for user j ||
|$b^{(j)}$     |  scalar; parameter for user j ||
| $\mathbf{x}^{(i)}$ |   vector; feature ratings for movie i        ||
| $n_u$        | number of users |num_users|
| $n_m$        | number of movies | num_movies |
| $n$          | number of features | num_features                    |
| $\mathbf{X}$ |  matrix of vectors $\mathbf{x}^{(i)}$         | X |
| $\mathbf{W}$ |  matrix of vectors $\mathbf{w}^{(j)}$         | W |
| $\mathbf{b}$ |  vector of bias parameters $b^{(j)}$ | b |
| $\mathbf{R}$ | matrix of elements $r(i,j)$                    | R |



## 2 - 推荐系统

![Image Name](https://cdn.kesci.com/upload/image/rgfu1mcgaa.png?imageView2/0/w/320/h/320)

在这份实验中，你将实现协作过滤学习算法，并将其应用于电影评级的数据集。
协同过滤推荐系统的目标是生成两个向量。对于每个用户，一个 "参数向量"，体现用户的电影品味。对于每部电影，都有一个相同大小的特征向量，体现了对电影的一些描述。这两个向量的点积加上偏置项应该可以产生一个对用户可能给该电影的评分的估计。

下图详细说明了这些向量是如何学习的。


![Image Name](https://cdn.kesci.com/upload/image/rgfu4494sd.PNG?imageView2/0/w/960/h/960)



现有的评级是以矩阵形式提供的，如图所示。$Y$包含评分；0.5到5（包括0.5级）。如果电影没有被评级，则为0。如果电影已经被评级，则$R$为1。电影在行中，用户在列中。每个用户都有一个参数向量$w^{user}$和偏置b。每部电影有一个特征向量$x^{movie}$。这些向量是通过使用现有的用户/电影评分作为训练数据同时学习的。一个训练例子如上所示：$\mathbf{w}^{(1)} \cdot \mathbf{x}^{(1)} + b^{(1)} = 4$。值得注意的是，特征向量$x^{movie}$必须满足所有用户，而用户向量$w^{user}$必须满足所有电影。这就是这个方法名称的来源--所有的用户合作生成评分集。


![Image Name](https://cdn.kesci.com/upload/image/rgfwruhxn3.PNG?imageView2/0/w/960/h/960)


一旦学会了特征向量和参数，就可以用来预测用户可能对一部未分级的电影进行评分。这在上图中显示。该方程是一个预测用户1对电影0的评分的例子。

在这个练习中，你将实现函数`cofiCostFunc`来计算协同过滤的目标函数。

实现目标函数后，你将使用TensorFlow自定义训练循环来学习协同过滤的参数。

第一步是详细说明实验中要使用的数据集和数据结构。

## 3 - 电影评分数据集

![Image Name](https://cdn.kesci.com/upload/image/rgfx25du3n.png?imageView2/0/w/320/h/320)

这个数据集来自 [MovieLens "ml-latest-small"](https://grouplens.org/datasets/movielens/latest/) 数据集。  
[F. Maxwell Harper and Joseph A. Konstan. 2015. The MovieLens Datasets: History and Context. ACM Transactions on Interactive Intelligent Systems (TiiS) 5, 4: 19:1–19:19. <https://doi.org/10.1145/2827872>]

原始数据集有9000部电影，由600个用户评分。该数据集的规模已经缩小，仅使用在2000年后的电影。这个数据集由0.5到5的评分组成，以0.5为单位递增。缩小后的数据集有$n_u = 443$用户，$n_m= 4778$电影。

下面，你将把电影数据集加载到变量$Y$和$R$中。

矩阵$Y$（一个$n_m \times n_u$的矩阵）存储了评分$y^{(i,j)}$。矩阵$R$是一个二值指标矩阵，如果用户$j$对电影$i$进行了评分，则$R(i,j)=1$，否则$R(i,j)=0$。

在这部分练习中，你还将使用矩阵，$\mathbf{X}$，$\mathbf{W}$和$\mathbf{b}$。

$$
\mathbf{X} = 
\begin{bmatrix}
--- (\mathbf{x}^{(0)})^T --- \\
--- (\mathbf{x}^{(1)})^T --- \\
\vdots \\
--- (\mathbf{x}^{(n_m-1)})^T --- \\
\end{bmatrix} , \quad
\mathbf{W} = 
\begin{bmatrix}
--- (\mathbf{w}^{(0)})^T --- \\
--- (\mathbf{w}^{(1)})^T --- \\
\vdots \\
--- (\mathbf{w}^{(n_u-1)})^T --- \\
\end{bmatrix},\quad
\mathbf{ b} = 
\begin{bmatrix}
 b^{(0)}  \\
 b^{(1)} \\
\vdots \\
b^{(n_u-1)} \\
\end{bmatrix}\quad
$$ 

$\mathbf{X}$的第$i$行对应于第$i$部电影的特征向量$x^{(i)}$。
特征向量$x^{(i)}$。$\mathbf{W}$ 的第j行对应于一个参数向量$\mathbf{w}^{(j)}$，用于第$j$个用户。$x^{(i)}$和$\mathbf{w}^{(j)}$都是$n$维的向量。

在本练习中，你将使用$n=10$，所以，$\mathbf{x}^{(i)}$和$\mathbf{w}^{(j)}$有10个元素。相应地，$\mathbf{X}$是一个$n_m \times 10$矩阵，$mathbf{W}$是一个$n_u \times 10$矩阵。

我们将从加载电影评级数据集开始，以了解数据的结构。
我们将加载$Y$和$R$从电影数据集中。 
我们还将加载$\mathbf{X}$、$\mathbf{W}$和$\mathbf{b}$的预计算值。这些数值将在实验的后面部分学习到，但我们将使用预先计算的数值来开发代价计算模型。

In [4]:
#Load data
X, W, b, num_movies, num_features, num_users = load_precalc_params_small()
Y, R = load_ratings_small()

print("Y", Y.shape, "R", R.shape)
print("X", X.shape)
print("W", W.shape)
print("b", b.shape)
print("num_features", num_features)
print("num_movies",   num_movies)
print("num_users",    num_users)

Y (4778, 443) R (4778, 443)
X (4778, 10)
W (443, 10)
b (1, 443)
num_features 10
num_movies 4778
num_users 443


In [8]:
#  From the matrix, we can compute statistics like average rating.
tsmean =  np.mean(Y[0, R[0, :].astype(bool)])
print(f"Average rating for movie 1 : {tsmean:0.3f} / 5" )

Average rating for movie 1 : 2.832 / 5


## 4 - 协同过滤学习算法


![Image Name](https://cdn.kesci.com/upload/image/rgfxlfdxr0.png?imageView2/0/w/320/h/320)


现在，你将开始实现协作过滤学习算法。你将从实现目标函数开始。

协同过滤算法在电影推荐的设置中考虑了一组$n$维的参数向量
$\mathbf{x}^{(0)},...,\mathbf{x}^{(n_m-1)}$, $\mathbf{w}^{(0)},...,\mathbf{w}^{(n_u-1)}$ 和 $b^{(0)}, ...,b^{(n_u-1)}$, 其中
模型预测用户$j$对电影$i$的评分为
$y^{(i,j)} = \mathbf{w}^{(j)}cdot \mathbf{x}^{(i)}+b^{(i)}$ 。给定一个数据集，其中包括一组由一些用户对一些电影的评分组成的数据集，你希望能够
学习参数向量$\mathbf{x}^{(0)},...,\mathbf{x}^{(n_m-1)},\mathbf{w}^{(0)},...,\mathbf{w}^{(n_u-1)}$ 和 $b^{(0)},...,b^{(n_u-1)}$ 能产生最佳拟合（最小化的平方误差）。

你将完成cofiCostFunc中的代码，以计算协同过滤的费用
协同过滤的代价函数。

### 4.1 协同过滤代价函数

协同过滤代价函数由下式给出
$$
J({\mathbf{x}^{(0)},...,\mathbf{x}^{(n_m-1)},\mathbf{w}^{(0)},b^{(0)},...,\mathbf{w}^{(n_u-1)},b^{(n_u-1)}})= \frac{1}{2}\sum_{(i,j):r(i,j)=1}(\mathbf{w}^{(j)} \cdot \mathbf{x}^{(i)} + b^{(j)} - y^{(i,j)})^2
+\underbrace{
\frac{\lambda}{2}
\sum_{j=0}^{n_u-1}\sum_{k=0}^{n-1}(\mathbf{w}^{(j)}_k)^2
+ \frac{\lambda}{2}\sum_{i=0}^{n_m-1}\sum_{k=0}^{n-1}(\mathbf{x}_k^{(i)})^2
}_{regularization}
\tag{1}
$$

（1）式中的第一个积分条件是对任意的$i$, $j$ 满足 $r(i,j)$ 等于 $1$，上式可以重写为下面的形式：

$$
= \frac{1}{2}\sum_{j=0}^{n_u-1} \sum_{i=0}^{n_m-1}r(i,j)*(\mathbf{w}^{(j)} \cdot \mathbf{x}^{(i)} + b^{(j)} - y^{(i,j)})^2
+\text{regularization}
$$

你现在将开始写cofiCostFunc (协同过滤代价函数) 来返回代价cost。

### 练习 1

**对于循环的实现：**   
首先使用for循环来实现成本函数。
考虑分两步来开发代价函数。首先，开发没有正则化的代价函数。下面提供了一个不包括正则化的测试案例来测试你的实现。一旦成功，添加正则化并运行包含正则化的测试。 请注意，只有在$R(i,j)=1$的情况下，你才应该积累用户$j$和电影$i$的成本。

In [5]:
def cofi_cost_func(X, W, b, Y, R, lambda_):
    n_m,n_u = Y.shape
    J = 0
    for j in range(n_u): 
        b_j = b[0,j]
        for i in range(n_m):
            y = Y[i,j]
            r = R[i,j]
            J += r * np.square(np.dot(W[j],X[i]) + b_j - y)
    J += lambda_* (np.sum(np.square(W)) + np.sum(np.square(X)))
    J = J/2
    return J

In [34]:
# GRADED FUNCTION: cofi_cost_func
# UNQ_C1

def cofi_cost_func(X, W, b, Y, R, lambda_):
    """
    Returns the cost for the content-based filtering
    Args:
      X (ndarray (num_movies,num_features)): matrix of item features
      W (ndarray (num_users,num_features)) : matrix of user parameters
      b (ndarray (1, num_users)            : vector of user parameters
      Y (ndarray (num_movies,num_users)    : matrix of user ratings of movies
      R (ndarray (num_movies,num_users)    : matrix, where R(i, j) = 1 if the i-th movies was rated by the j-th user
      lambda_ (float): regularization parameter
    Returns:
      J (float) : Cost
    """
    nm, nu = Y.shape
    J = 0
    ### START CODE HERE ###  
    for j in range(nu):
      w = W[j,:]
      b_j = b[0,j]
      for i in range(nm):
        x = X[i,:]
        y = Y[i,j]
        r = R[i,j]
        J += np.square(r * (np.dot(w,x) + b_j - y ) )
    J += lambda_* (np.sum(np.square(W)) + np.sum(np.square(X)))
    J = J/2
    ### END CODE HERE ### 

    return J

In [11]:
# Public tests
from public_tests import *
test_cofi_cost_func(cofi_cost_func)

[✓] 无正则化代价: 37.9550
[✓] 有正则化代价: 60.0263
[✓] 返回值类型正确（标量）

所有测试通过！cofi_cost_func 实现正确 ✅


你可以在两个for循环中构建代码，类似于（1）中的求和。先执行没有正则化的代码。  
注意(1)中的一些元素是向量。使用np.dot()。你也可以使用np.square()。
密切注意哪些元素是以i为索引的，哪些是以j为索引的，别忘了除以2。
    
```python     
    ### START CODE HERE ###  
    for j in range(nu):
        
        
        for i in range(nm):
            
            
    ### END CODE HERE ### 
```    

下面是一些更多的细节。下面的代码在使用前从矩阵中拉出每个元素。也可以直接引用矩阵。 
这段代码不包含正则化。
    
```python 
    nm,nu = Y.shape
    J = 0
    ### START CODE HERE ###  
    for j in range(nu):
        w = W[j,:]
        b_j = b[0,j]
        for i in range(nm):
            x = 
            y = 
            r =
            J += 
    J = J/2
    ### END CODE HERE ### 

```
    
    
```python 
    nm,nu = Y.shape
    J = 0
    ### START CODE HERE ###  
    for j in range(nu):
        w = W[j,:]
        b_j = b[0,j]
        for i in range(nm):
            x = X[i,:]
            y = Y[i,j]
            r = R[i,j]
            J += np.square(r * (np.dot(w,x) + b_j - y ) )
    J = J/2
    ### END CODE HERE ### 
```
    
正则化只是对W数组和X数组中的每个元素进行平方，然后将所有的平方元素相加。
你可以利用np.square()和np.sum()来实现，正则化细节
    
```python 
    J += lambda_* (np.sum(np.square(W)) + np.sum(np.square(X)))
```


    


In [36]:
# Reduce the data set size so that this runs faster
num_users_r = 4
num_movies_r = 5 
num_features_r = 3

X_r = X[:num_movies_r, :num_features_r]
W_r = W[:num_users_r,  :num_features_r]
b_r = b[0, :num_users_r].reshape(1,-1)
Y_r = Y[:num_movies_r, :num_users_r]
R_r = R[:num_movies_r, :num_users_r]

# Evaluate cost function
J = cofi_cost_func(X_r, W_r, b_r, Y_r, R_r, 0);
print(f"Cost: {J:0.2f}")

Cost: 13.67


**期望输出 (lambda = 0)**:  
$13.67$.

In [37]:
# Evaluate cost function with regularization 
J = cofi_cost_func(X_r, W_r, b_r, Y_r, R_r, 1.5);
print(f"Cost (with regularization): {J:0.2f}")

Cost (with regularization): 28.09


**期望输出**:

28.09

**矢量化实现**

创建一个矢量化的实现来计算$J$是很重要的，因为它以后会在优化过程中被多次调用。所利用的线性代数不是本系列的重点，所以提供了实现。如果你是线性代数方面的专家，可以自由地创建你的版本，而不必参考下面的代码。

运行下面的代码，并验证其产生的结果与非矢量化版本相同。

In [38]:
def cofi_cost_func_v(X, W, b, Y, R, lambda_):
    """
    Returns the cost for the content-based filtering
    Vectorized for speed. Uses tensorflow operations to be compatible with custom training loop.
    Args:
      X (ndarray (num_movies,num_features)): matrix of item features
      W (ndarray (num_users,num_features)) : matrix of user parameters
      b (ndarray (1, num_users)            : vector of user parameters
      Y (ndarray (num_movies,num_users)    : matrix of user ratings of movies
      R (ndarray (num_movies,num_users)    : matrix, where R(i, j) = 1 if the i-th movies was rated by the j-th user
      lambda_ (float): regularization parameter
    Returns:
      J (float) : Cost
    """
    j = (tf.linalg.matmul(X, tf.transpose(W)) + b - Y)*R
    J = 0.5 * tf.reduce_sum(j**2) + (lambda_/2) * (tf.reduce_sum(X**2) + tf.reduce_sum(W**2))
    return J

In [39]:
# Evaluate cost function
J = cofi_cost_func_v(X_r, W_r, b_r, Y_r, R_r, 0);
print(f"Cost: {J.numpy():0.2f}")

# Evaluate cost function with regularization 
J = cofi_cost_func_v(X_r, W_r, b_r, Y_r, R_r, 1.5);
print(f"Cost (with regularization): {J.numpy():0.2f}")

Cost: 13.67
Cost (with regularization): 28.09


**Expected Output**:  
Cost: 13.67  
Cost (with regularization): 28.09

## 5 - 学习电影推荐

![Image Name](https://cdn.kesci.com/upload/image/rgfzb4up5m.png?imageView2/0/w/320/h/320)

在你完成了协同过滤成本函数的实现之后，你就可以开始训练你的算法，为自己推荐电影。

在下面的单元格中，你可以输入你自己的电影选择。然后，该算法将为你做出推荐! 我们根据自己的喜好填写了一些数值，但在你用我们的选择工作之后，你应该改变这些数值以符合你的口味。
数据集中所有电影的列表在[movie list](data/small_movie_list.csv)文件中。

In [40]:
movieList, movieList_df = load_Movie_List_pd()

my_ratings = np.zeros(num_movies)          #  Initialize my ratings

# Check the file small_movie_list.csv for id of each movie in our dataset
# For example, Toy Story 3 (2010) has ID 2700, so to rate it "5", you can set
my_ratings[2700] = 5 

#Or suppose you did not enjoy Persuasion (2007), you can set
my_ratings[2609] = 2;

# We have selected a few movies we liked / did not like and the ratings we
# gave are as follows:
my_ratings[929]  = 5   # Lord of the Rings: The Return of the King, The
my_ratings[246]  = 5   # Shrek (2001)
my_ratings[2716] = 3   # Inception
my_ratings[1150] = 5   # Incredibles, The (2004)
my_ratings[382]  = 2   # Amelie (Fabuleux destin d'Amélie Poulain, Le)
my_ratings[366]  = 5   # Harry Potter and the Sorcerer's Stone (a.k.a. Harry Potter and the Philosopher's Stone) (2001)
my_ratings[622]  = 5   # Harry Potter and the Chamber of Secrets (2002)
my_ratings[988]  = 3   # Eternal Sunshine of the Spotless Mind (2004)
my_ratings[2925] = 1   # Louis Theroux: Law & Disorder (2008)
my_ratings[2937] = 1   # Nothing to Declare (Rien à déclarer)
my_ratings[793]  = 5   # Pirates of the Caribbean: The Curse of the Black Pearl (2003)
my_rated = [i for i in range(len(my_ratings)) if my_ratings[i] > 0]

print('\nNew user ratings:\n')
for i in range(len(my_ratings)):
    if my_ratings[i] > 0 :
        print(f'Rated {my_ratings[i]} for  {movieList_df.loc[i,"title"]}');


New user ratings:

Rated 5.0 for  Shrek (2001)
Rated 5.0 for  Harry Potter and the Sorcerer's Stone (a.k.a. Harry Potter and the Philosopher's Stone) (2001)
Rated 2.0 for  Amelie (Fabuleux destin d'Amélie Poulain, Le) (2001)
Rated 5.0 for  Harry Potter and the Chamber of Secrets (2002)
Rated 5.0 for  Pirates of the Caribbean: The Curse of the Black Pearl (2003)
Rated 5.0 for  Lord of the Rings: The Return of the King, The (2003)
Rated 3.0 for  Eternal Sunshine of the Spotless Mind (2004)
Rated 5.0 for  Incredibles, The (2004)
Rated 2.0 for  Persuasion (2007)
Rated 5.0 for  Toy Story 3 (2010)
Rated 3.0 for  Inception (2010)
Rated 1.0 for  Louis Theroux: Law & Disorder (2008)
Rated 1.0 for  Nothing to Declare (Rien à déclarer) (2010)


现在，让我们把这些评论加到$Y$和$R$中，并把评分标准化。

In [41]:
# Reload ratings and add new ratings
Y, R = load_ratings_small()
Y    = np.c_[my_ratings, Y]
R    = np.c_[(my_ratings != 0).astype(int), R]

# Normalize the Dataset
Ynorm, Ymean = normalizeRatings(Y, R)

让我们准备训练这个模型。初始化参数并选择Adam优化器。

In [42]:
#  Useful Values
num_movies, num_users = Y.shape
num_features = 100

# Set Initial Parameters (W, X), use tf.Variable to track these variables
tf.random.set_random_seed(1234) # for consistent results
W = tf.Variable(tf.random.normal((num_users,  num_features),dtype=tf.float64),  name='W')
X = tf.Variable(tf.random.normal((num_movies, num_features),dtype=tf.float64),  name='X')
b = tf.Variable(tf.random.normal((1,          num_users),   dtype=tf.float64),  name='b')

# Instantiate an optimizer.
# optimizer = tf.keras.optimizers.Adam(lr=1e-1)
optimizer = tf.compat.v1.train.AdamOptimizer(learning_rate=0.001)

现在让我们来训练这个协同过滤模型，它将学习参数$\mathbf{X}$， $\mathbf{W}$和$\mathbf{b}$. 


同时学习$w$、$b$和$x$所涉及的操作并不属于TensorFlow神经网络包中提供的典型 "层"。 因此，课程2中使用的流程：Model, Compile(), Fit(), Predict()，并不直接适用。相反，我们可以使用一个自定义的训练循环。

回忆一下前面的实验室，梯度下降的步骤：
- 重复直到收敛：
    - 计算前向通道
    - 计算损失相对于参数的导数
    - 使用学习率和计算的导数更新参数 
    
TensorFlow具有为你计算导数的奇妙能力。这显示在下面。在`tf.GradientTape()`部分，对Tensorflow变量的操作被跟踪。当后来调用`tape.gradient()`时，它将返回相对于被跟踪变量的损失梯度。然后可以使用优化器将梯度应用于参数。
这是对TensorFlow和其他机器学习框架的一个有用功能的非常简单的介绍。更多的信息可以通过调查感兴趣的框架中的 "自定义训练循环 "找到。
    


In [43]:
iterations = 200
lambda_ = 1
for iter in range(iterations):
    # Use TensorFlow’s GradientTape
    # to record the operations used to compute the cost 
    with tf.GradientTape() as tape:

        # Compute the cost (forward pass included in cost)
        cost_value = cofi_cost_func_v(X, W, b, Ynorm, R, lambda_)

    # Use the gradient tape to automatically retrieve
    # the gradients of the trainable variables with respect to the loss
    grads = tape.gradient( cost_value, [X,W,b] )

    # Run one step of gradient descent by updating
    # the value of the variables to minimize the loss.
    optimizer.apply_gradients( zip(grads, [X,W,b]) )

    # Log periodically.
    if iter % 20 == 0:
        print(f"Training loss at iteration {iter}: {cost_value:0.1f}")

Training loss at iteration 0: 2321191.3
Training loss at iteration 20: 2065538.2
Training loss at iteration 40: 1843968.6
Training loss at iteration 60: 1653776.5
Training loss at iteration 80: 1490020.5
Training loss at iteration 100: 1348156.0
Training loss at iteration 120: 1224523.2
Training loss at iteration 140: 1116216.4
Training loss at iteration 160: 1020904.6
Training loss at iteration 180: 936693.6


## 6 - 推荐
下面，我们计算出所有电影和用户的评分，并显示被推荐的电影。这些是基于上面输入的`my_ratings[]`的电影和评分。为了预测用户$j$对电影$i$的评分，你要计算$\mathbf{w}^{(j)} \cdot \mathbf{x}^{(i)} + b^{(j)}$。这可以用矩阵乘法对所有评级进行计算。

In [44]:
# Make a prediction using trained weights and biases
p = np.matmul(X.numpy(), np.transpose(W.numpy())) + b.numpy()

#restore the mean
pm = p + Ymean

my_predictions = pm[:,0]

# sort predictions
ix = tf.argsort(my_predictions, direction='DESCENDING')

for i in range(17):
    j = ix[i]
    if j not in my_rated:
        print(f'Predicting rating {my_predictions[j]:0.2f} for movie {movieList[j]}')

print('\n\nOriginal vs Predicted ratings:\n')
for i in range(len(my_ratings)):
    if my_ratings[i] > 0:
        print(f'Original {my_ratings[i]}, Predicted {my_predictions[i]:0.2f} for {movieList[i]}')

Predicting rating 36.41 for movie Margin Call (2011)
Predicting rating 33.43 for movie Fuck You, Goethe (Fack Ju Göhte) (2013)
Predicting rating 33.07 for movie Divine Secrets of the Ya-Ya Sisterhood (2002)
Predicting rating 32.75 for movie Asterix and the Vikings (Astérix et les Vikings) (2006)
Predicting rating 32.51 for movie Giver, The (2014)
Predicting rating 32.27 for movie Samsara (2011)
Predicting rating 32.27 for movie O Brother, Where Art Thou? (2000)
Predicting rating 32.07 for movie Bigger, Stronger, Faster* (2008)
Predicting rating 32.07 for movie Namesake, The (2006)
Predicting rating 32.00 for movie Asterix: The Land of the Gods (Astérix: Le domaine des dieux) (2014)
Predicting rating 30.95 for movie Powder Blue (2009)
Predicting rating 30.77 for movie Kingsman: The Secret Service (2015)
Predicting rating 30.71 for movie Louis C.K.: Oh My God (2013)
Predicting rating 30.20 for movie The Meyerowitz Stories (2017)
Predicting rating 29.76 for movie League of Ordinary Gentle

在实践中，可以利用额外的信息来加强我们的预测。以上，前几百部电影的预测评分位于一个小范围内。我们可以通过从那些顶级电影、平均收视率高的电影和收视率超过20的电影中进行选择来增强上述内容。本节使用了一个[Pandas](https://pandas.pydata.org/)的数据框架，它有很多方便的排序功能。

In [45]:
filter=(movieList_df["number of ratings"] > 20)
movieList_df["pred"] = my_predictions
movieList_df = movieList_df.reindex(columns=["pred", "mean rating", "number of ratings", "title"])
movieList_df.loc[ix[:300]].loc[filter].sort_values("mean rating", ascending=False)

,pred,mean rating,number of ratings,title
395,23.677167,4.000000,123,"Beautiful Mind, A (2001)"
3862,30.765551,3.986111,36,Kingsman: The Secret Service (2015)
2756,19.094244,3.954545,22,"Town, The (2010)"
451,21.206042,3.952381,21,And Your Mother Too (Y tu mamá también) (2001)
642,18.163626,3.945652,46,Adaptation (2002)
2314,26.528066,3.945652,46,Gran Torino (2008)
916,22.404594,3.933333,30,Battle Royale (Batoru rowaiaru) (2000)
3556,22.078910,3.916667,54,"Wolf of Wall Street, The (2013)"
132,19.579884,3.851351,37,Billy Elliot (2000)
3665,22.021369,3.833333,30,X-Men: Days of Future Past (2014)


## 7 - 祝贺!


![Image Name](https://cdn.kesci.com/upload/image/rgg0ird0mc.png?imageView2/0/w/320/h/320)

你已经实现了一个有用的推荐系统！